# 🔍 SmartLens — Contextual Image Caption Generator

**Model:** BLIP (Bootstrapping Language-Image Pre-training) by Salesforce  
**Features:**
- 🖼️ Generates 3 diverse captions per image using beam search
- 🌐 Translates best caption to Hindi
- 📊 BLEU score evaluation
- 🚀 Runs fully offline — no API key needed

---

## 1. Install Dependencies

In [ ]:
# Run this cell once to install all required packages
!pip install transformers torch torchvision Pillow gradio deep-translator nltk -q

## 2. Load BLIP Model

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
import requests
from io import BytesIO

MODEL_NAME = "Salesforce/blip-image-captioning-base"

print("Loading model (downloads ~1GB on first run, cached afterwards)...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"✅ Model ready on: {device}")

## 3. Caption Generation Functions

In [ ]:
def generate_captions(image: Image.Image, num_captions: int = 3) -> list:
    """
    Generate multiple diverse captions using greedy decoding + diverse beam search.
    
    Args:
        image: PIL Image object
        num_captions: number of captions to return (default 3)
    Returns:
        List of caption strings
    """
    image = image.convert("RGB")
    inputs = processor(image, return_tensors="pt").to(device)
    captions = []

    # Greedy: most confident single caption
    with torch.no_grad():
        greedy_ids = model.generate(**inputs, max_new_tokens=50)
    captions.append(processor.decode(greedy_ids[0], skip_special_tokens=True))

    # Diverse beam search for alternatives
    with torch.no_grad():
        beam_ids = model.generate(
            **inputs,
            num_beams=6,
            num_return_sequences=2,
            max_new_tokens=60,
            diversity_penalty=1.2,
            num_beam_groups=2,
        )
    for beam_id in beam_ids:
        cap = processor.decode(beam_id, skip_special_tokens=True)
        if cap not in captions:
            captions.append(cap)

    return captions[:num_captions]


def translate_to_hindi(text: str) -> str:
    """Translate English caption to Hindi (free, no API key)."""
    from deep_translator import GoogleTranslator
    return GoogleTranslator(source="en", target="hi").translate(text)


def load_image_from_url(url: str) -> Image.Image:
    """Helper to load an image from a URL."""
    response = requests.get(url)
    return Image.open(BytesIO(response.content))

print("✅ Functions defined")

## 4. Test with a Sample Image

In [ ]:
import matplotlib.pyplot as plt

# Load a test image (dog on beach — classic CV test image)
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
image = load_image_from_url(img_url)

# Show image
plt.figure(figsize=(6, 4))
plt.imshow(image)
plt.axis("off")
plt.title("Input Image")
plt.show()

# Generate captions
print("\nGenerating captions...")
captions = generate_captions(image, num_captions=3)

print("\n🖼️  Generated Captions:")
for i, cap in enumerate(captions, 1):
    print(f"  [{i}] {cap}")

# Translate best caption
print("\n🌐 Hindi Translation (Best Caption):")
hindi = translate_to_hindi(captions[0])
print(f"  {hindi}")

## 5. BLEU Score Evaluation

BLEU (Bilingual Evaluation Understudy) measures how similar a generated caption is to a reference caption. Score ranges from 0 to 1 (higher = better).

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def compute_bleu(reference: str, hypothesis: str) -> float:
    """
    Compute BLEU-4 score between a reference caption and generated caption.
    
    Args:
        reference: ground truth caption
        hypothesis: model-generated caption
    Returns:
        BLEU score (0 to 1)
    """
    ref_tokens = nltk.word_tokenize(reference.lower())
    hyp_tokens = nltk.word_tokenize(hypothesis.lower())
    smoother = SmoothingFunction().method1
    score = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)
    return round(score, 4)


# Example: evaluate against a manually written reference caption
reference_caption = "a dog sitting on the grass in a park"
generated_caption = captions[0]

bleu = compute_bleu(reference_caption, generated_caption)

print(f"Reference : {reference_caption}")
print(f"Generated : {generated_caption}")
print(f"BLEU-4 Score: {bleu}  (higher is better, max = 1.0)")

## 6. Batch Evaluation on Multiple Images

In [ ]:
import pandas as pd

# Sample dataset: (image_url, reference_caption)
test_set = [
    (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg",
        "a dog sitting on the grass"
    ),
    (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
        "a cat lying on a surface"
    ),
    (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b9/Above_Gotham.jpg/320px-Above_Gotham.jpg",
        "an aerial view of a city with tall buildings"
    ),
]

results = []
for url, ref in test_set:
    try:
        img = load_image_from_url(url)
        caps = generate_captions(img, num_captions=1)
        gen = caps[0]
        bleu = compute_bleu(ref, gen)
        results.append({"Reference": ref, "Generated": gen, "BLEU-4": bleu})
        print(f"✓ Processed: {url.split('/')[-1]}")
    except Exception as e:
        print(f"✗ Failed: {e}")

df = pd.DataFrame(results)
print("\n📊 Evaluation Results:")
display(df)
print(f"\nAverage BLEU-4: {df['BLEU-4'].mean():.4f}")

## 7. Launch Gradio Web UI (Optional)

Run the cell below to launch a full interactive web interface.

In [ ]:
# Uncomment and run to launch web UI
# import subprocess
# subprocess.Popen(["python", "app.py"])
# print("App running at http://localhost:7860")